# Arm 2 minimal-baseline sweep - Colab sandbox

**What this does.** Scores general-purpose VLMs on one task, offline, against real logged Track 4
data: *given the raw 640x480 camera frame and the instruction (`go_green` ...), point at the target
marker*. The model's pixel answer is converted to millimetres with a per-frame ArUco homography
(scorer only - the model never sees a rectified image) and compared with the logged target.
**4 candidates x 3 prompt variants** (`baseline`, `oriented`, `oriented_aruco`), same 60 frames
(6 per session x 10 sessions) as the local Qwen run, 20 mm tolerance.

**Run order** - run cells top to bottom; each `##` cell is one step.

| # | Cell | Time (T4) | Stops the run if... |
|---|------|-----------|---------------------|
| 0 | Config | - | - |
| 1 | Setup (GPU, Drive, installs, unzip, imports) | 2-4 min | no GPU / bundle missing |
| 2 | HF auth + gated-model probe | seconds | never (PaliGemma2 is just skipped) |
| 3 | Prompts (+ your overrides) | seconds | bad override template |
| 4 | Frames (decode + ArUco homography) | ~1 min | sampled frames differ from the local run |
| 5 | Candidate table | seconds | - |
| 6 | **Smoke** (1-2 frames/candidate, downloads + loads every model) | 10-25 min (downloads) | never - reports which candidates work |
| 7 | **Pipeline validation** (Qwen vs local numbers) | ~5 min | Colab's Qwen does not reproduce the local run |
| 8 | **Full sweep** (checkpointed - safe to interrupt and re-run) | 30-90 min | never |
| 9 | Results (table, diagnostics, per-session, export) | seconds | - |

**Checkpointing.** Every single model call is saved to a JSON file on Drive before the next one
starts. If the runtime dies, re-run from the top: finished items are skipped, and a finished
candidate is skipped without even loading its model. Change `RUN_LABEL` to start clean.

**Read this before trusting any number** (details in `docs/ARM2_MINIMAL_BASELINE_MULTI_CANDIDATE.md`):
the 2026-09-18 local scorer had two bugs (wrong ground-truth frame; Qwen's answers read in the wrong
pixel space) that made a 67 %-hit result look like 0 %. This notebook uses the corrected scorer and
also reports the diagnostics that would expose a wrong coordinate assumption for the three
candidates that have never run. Three of four backends have **never executed for real** before this
notebook - failures in smoke mode are expected information, not a broken notebook.

## 0. Config - the cells you are meant to edit

In [ ]:
# @title 0. Config (edit here; nothing below needs touching for a normal run)

# ---- where things live -------------------------------------------------------------------
DRIVE_ROOT  = "/content/drive/MyDrive/vri2026_track4_bootstrap"      # same Drive folder as the other Colab notebooks
BUNDLE_ZIP  = f"{DRIVE_ROOT}/arm2_sweep_bundle.zip"                  # built by stage_arm2_sweep_for_colab.py
RESULTS_DIR = f"{DRIVE_ROOT}/arm2_sweep_results"                     # checkpoints + final JSON land here (Drive-backed)
RUN_LABEL   = "run1"        # same label = RESUME that checkpoint; new label = fresh run

# ---- experiment knobs ---------------------------------------------------------------------
FRAMES_PER_SESSION = 6      # 6 = identical frames to the local run (needed for the validation gate)
TOLERANCE_MM       = 20.0   # hit threshold; recomputed at aggregation, so changing it needs no re-run
MAX_NEW_TOKENS     = 24     # local run used 24; part of each result's identity
DTYPE              = "auto" # auto = bf16 on A100/L4/Ampere+, fp16 on T4 (no native bf16). Or force "bf16" / "fp16" / "fp32".
SMOKE_FRAMES       = 2      # frames per candidate in the smoke stage
PROMPT_VARIANTS_TO_RUN = ["baseline", "oriented", "oriented_aruco"]   # add your own override names here (cell 3)
CANDIDATES_TO_RUN  = None   # None = all; or e.g. ["qwen2_5_vl_3b", "moondream2:point"]  (keys are printed in cell 5)

# ---- environment --------------------------------------------------------------------------
# Moondream2's remote code (README-recommended revision) breaks on transformers>=5; the local Qwen
# run used 5.17.0. "<5" makes every candidate loadable in one session; the validation gate (cell 7)
# tells you whether Qwen still matches the local numbers under it.
TRANSFORMERS_SPEC = "transformers>=4.56,<5"
INSTALL_DEPS = True
USE_MOCK_BACKENDS = False   # True = fake models (no GPU/downloads) to test the notebook plumbing itself
FORCE_CONTINUE_IF_VALIDATION_FAILS = False   # leave False: a failed gate means a Colab-side bug

## 1. Setup

In [ ]:
# @title 1. Setup: GPU, Drive, installs, unzip, imports
import importlib.util, os, shutil, subprocess, sys, tempfile, time, zipfile

IN_COLAB = importlib.util.find_spec("google.colab") is not None
try:
    display
except NameError:
    display = lambda x: print(x.to_string() if hasattr(x, "to_string") else x)

def sh(*args):
    print("$", " ".join(args))
    r = subprocess.run(list(args), capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:])
        raise RuntimeError(f"command failed ({r.returncode}): {' '.join(args)}")
    return r

# --- GPU -----------------------------------------------------------------------------------
if not USE_MOCK_BACKENDS:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU (or better), then re-run.")
    print("GPU:", torch.cuda.get_device_name(0), "| capability", torch.cuda.get_device_capability(0),
          "| %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

# --- Drive + bundle --------------------------------------------------------------------------
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("(not in Colab: skipping Drive mount and pip installs; assuming dependencies are present)")
os.makedirs(RESULTS_DIR, exist_ok=True)
if not os.path.exists(BUNDLE_ZIP):
    raise FileNotFoundError(f"Upload arm2_sweep_bundle.zip to {DRIVE_ROOT}/ first (path checked: {BUNDLE_ZIP}).")

# --- pip -------------------------------------------------------------------------------------
if IN_COLAB and INSTALL_DEPS:
    pre_imported = "transformers" in sys.modules
    sh(sys.executable, "-m", "pip", "install", "-q", TRANSFORMERS_SPEC, "accelerate", "qwen-vl-utils", "av",
       "huggingface_hub", "pandas")
    import cv2
    if not hasattr(cv2, "aruco") or not hasattr(cv2.aruco, "ArucoDetector"):
        sh(sys.executable, "-m", "pip", "install", "-q", "opencv-contrib-python-headless")
        print("!! Installed opencv-contrib (needed for ArUco). RESTART the session (Runtime > Restart) and re-run from the top.")
    if pre_imported:
        print("!! transformers was already imported before pip ran. RESTART the session and re-run from the top so the pinned version is used.")

# --- unzip + imports -------------------------------------------------------------------------
BUNDLE_PARENT = "/content" if IN_COLAB else tempfile.mkdtemp(prefix="arm2_colab_")
BUNDLE_DIR = os.path.join(BUNDLE_PARENT, "arm2_bundle")
if os.path.exists(BUNDLE_DIR):
    shutil.rmtree(BUNDLE_DIR)
t0 = time.time()
with zipfile.ZipFile(BUNDLE_ZIP) as zf:
    zf.extractall(BUNDLE_PARENT)
print(f"extracted bundle to {BUNDLE_DIR} in {time.time() - t0:.0f}s")
for p in (os.path.join(BUNDLE_DIR, "host_software"), BUNDLE_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda v: f"{v:.2f}")
from ml_jetson_vla.core import minimal_vlm_policy as mvp
from ml_jetson_vla.core.vlm_backends import resolve_torch_dtype, MOONDREAM2_PINNED_REVISION
from ml_jetson_vla.deployment import colab_sweep as cs
assert cs.__file__.startswith(BUNDLE_DIR), f"imported repo code from the wrong place: {cs.__file__}"

BRONZE_DIR = os.path.join(BUNDLE_DIR, "host_software", "data", "01_bronze")
REFERENCE_JSON = os.path.join(BUNDLE_DIR, "reference", "arm2_local_qwen_20260918_RESCORED_v2.json")
ENV_META = cs.collect_env_meta()
print("\nenvironment:", ENV_META)
if not USE_MOCK_BACKENDS:
    print("\nDTYPE =", DTYPE, "->", resolve_torch_dtype(DTYPE)[1],
          "(the local run was bf16 on CPU; on a T4 this becomes fp16, which can differ numerically and is the"
          " first suspect if the validation gate in cell 7 fails - retry with DTYPE='bf16', slower but numerically closer)")

## 2. Hugging Face auth (PaliGemma2 is gated)

In [ ]:
# @title 2. HF auth + gated-model probe
GATED_REPO = "google/paligemma2-3b-mix-448"
HF_TOKEN, how = cs.get_hf_token()
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("HF auth via", how)
else:
    print(f"No HF token ({how}).\n"
          "PaliGemma2 is a GATED model: to include it, (1) accept the Gemma license at "
          f"https://huggingface.co/{GATED_REPO} , (2) create a read token at https://huggingface.co/settings/tokens , "
          "(3) add it as a Colab secret named HF_TOKEN (key icon, left sidebar) with 'Notebook access' ON, then re-run "
          "this cell. Without it the two PaliGemma2 candidates are skipped and everything else still runs.")
if not USE_MOCK_BACKENDS:
    ok, msg = cs.check_hf_access(GATED_REPO, HF_TOKEN)
    print(("GATED ACCESS OK: " if ok else "GATED ACCESS UNAVAILABLE -> PaliGemma2 candidates will be skipped.\n") + msg)

## 3. Prompts

Three built-in variants: **`baseline`** (bare), **`oriented`** (adds platform/task context - note the
2026-09-18 conclusion that it "helped" was an artifact of the scoring bugs, see the results doc), and
**`oriented_aruco`** (oriented + the real ArUco marker layout generated from `ground_truth_manifest.json`,
prompt text only - the model still gets the raw frame). Use the override cell to try your own wording
without re-zipping.

In [ ]:
# @title 3a. Print the built-in prompts exactly as the model will see them (go_green example)
for name in cs.BUILTIN_VARIANTS:
    text, label, _ = mvp.build_prompt("go_green", 640, 480, name)
    print(f"----- {name}  ({len(text)} chars) -----\n{text}\n")

In [ ]:
# @title 3b. Your prompt overrides (edit + re-run this cell; no re-zip needed)
# Fields available in a template: {target_label} {width} {height} {platform_w_mm} {platform_h_mm}
# The JSON example needs DOUBLED braces. The three built-in names cannot be overridden - use a new name.
PROMPT_OVERRIDES = {
    # "concise": (
    #     "Find the {target_label} in this {width}x{height} image. Answer with ONLY "
    #     '{{"target_point_xy": [x, y]}} in pixel coordinates.'
    # ),
}
if PROMPT_OVERRIDES:
    registered = cs.register_prompt_overrides(PROMPT_OVERRIDES)
    for n in registered:
        if n not in PROMPT_VARIANTS_TO_RUN:
            PROMPT_VARIANTS_TO_RUN.append(n)
    print("registered + added to PROMPT_VARIANTS_TO_RUN:", registered)
print("variants that will run:", PROMPT_VARIANTS_TO_RUN)

## 4. Frames

In [ ]:
# @title 4. Decode the sampled frames + ArUco homographies (same sampling as the local run)
t0 = time.time()
frames, skipped = cs.prepare_frames(BRONZE_DIR, FRAMES_PER_SESSION)
print(f"{len(frames)} frames ready in {time.time()-t0:.0f}s; skipped: {skipped or 'none'}")
if FRAMES_PER_SESSION == 6:
    parity = cs.sampling_parity(frames, REFERENCE_JSON)
    print("sampling parity with the local run:", parity)
    assert parity["identical"], "Colab sampled DIFFERENT frames than the local run - results would not be comparable."
else:
    print(f"FRAMES_PER_SESSION={FRAMES_PER_SESSION}: not the local run's 6, so cell 7's validation gate is skipped.")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].imshow(frames[0].frame_bgr[..., ::-1]); ax[0].set_title(f"raw frame as the model sees it\n{frames[0].ident[-24:]} ({frames[0].instruction})")
ax[1].imshow(cs.render_overlay(frames[0], None)); ax[1].set_title("green circle = logged target (via homography)")
for a in ax: a.axis("off")
plt.show()

## 5. Candidates

In [ ]:
# @title 5. Candidate table (edit specs here: revision, mode, kwargs, or add/remove candidates)
if USE_MOCK_BACKENDS:
    specs = cs.mock_candidate_specs()
else:
    specs = cs.default_candidate_specs(max_new_tokens=MAX_NEW_TOKENS, dtype=DTYPE)
    # Optional extra (NOT the requested InternVL2.5; it is InternVL3.5-4B via native transformers) - uncomment
    # to have a fallback if InternVL2.5's remote code fails to load on this transformers version:
    # specs.append(cs.CandidateSpec("internvl3_5_4b_hf", "internvl3_5_4b_hf", dict(max_new_tokens=MAX_NEW_TOKENS, dtype=DTYPE)))
    # Example: Moondream2 in fp16 instead of the README's default: cs.CandidateSpec(..., dict(mode="query", dtype="fp16"))
if CANDIDATES_TO_RUN is not None:
    specs = [s for s in specs if s.key in set(CANDIDATES_TO_RUN)]
display(pd.DataFrame([{"key": s.key, "backend": s.backend, "variants": s.variants or "all", "gated": bool(s.hf_gated_repo),
                       "kwargs": s.kwargs, "note": s.note} for s in specs]))
print("Moondream2 pinned revision:", MOONDREAM2_PINNED_REVISION, "(tag 2025-06-21, the README-recommended one)")
os.makedirs(RESULTS_DIR, exist_ok=True)

## 6. Smoke mode (run this first)

Loads **every** candidate and runs 1-2 frames x every prompt variant, with the **raw model text** shown,
so a format problem (prose instead of JSON, a different coordinate convention) is visible immediately. A
candidate that fails to load or answers unparseably is reported here and the sweep continues without it.
Downloads happen here (cached for the rest of the session). Results go to a *separate* checkpoint.

In [ ]:
# @title 6. Smoke: 1-2 frames per candidate (separate checkpoint)
smoke_store = cs.CheckpointStore(os.path.join(RESULTS_DIR, f"smoke_{RUN_LABEL}.json"), meta=ENV_META)
smoke_frames = cs.pick_smoke_frames(frames, SMOKE_FRAMES)
print("smoke frames:", [f.ident[-22:] + " " + f.instruction for f in smoke_frames])
try:
    smoke_status = cs.run_sweep(specs, smoke_frames, PROMPT_VARIANTS_TO_RUN, smoke_store, tolerance_mm=TOLERANCE_MM,
                                max_new_tokens=MAX_NEW_TOKENS, hf_token=HF_TOKEN)
except KeyboardInterrupt:
    print("Interrupted - progress is saved; re-run this cell to resume.")
    raise
print("\n===== SMOKE REPORT =====")
display(cs.smoke_report(smoke_store, specs, smoke_frames, PROMPT_VARIANTS_TO_RUN, MAX_NEW_TOKENS))
print("\nstatus per candidate:", {k: v for k, v in smoke_status.items()})
if smoke_store.errors:
    print("\nFirst error per candidate (full tracebacks are in the smoke checkpoint JSON):")
    seen = set()
    for e in smoke_store.errors:
        if e["candidate"] not in seen:
            seen.add(e["candidate"]); print(f"  [{e['candidate']}] {e['stage']}: {e['error_type']}: {e['error'][:300]}")
RUNNABLE = [s.key for s in specs if smoke_status.get(s.key) in ("complete", "nothing_to_do") and
            any(e.get("parse_ok") for k, e in smoke_store.items.items() if e["candidate"] == s.key)]
print("\nwill run in the full sweep:", RUNNABLE, "\nskipped (failed/unparseable/gated):", [s.key for s in specs if s.key not in RUNNABLE])

## 7. Pipeline validation (Qwen must reproduce the local run)

The local run (Qwen2.5-VL-3B, CPU, bf16) gave, on these exact 60 frames: **legacy** mean error
183.8 mm (baseline) / 170.3 mm (oriented) - the historical number, wrong-frame and read as raw pixels,
which depends only on the model's raw outputs - and, after correcting both scoring bugs, **27.9 mm /
40 of 60 hits** (baseline) and **43.7 mm / 21 hits** (oriented). Colab should land close on all of them,
and - the sharpest check - its per-frame pixel answers should agree with the local ones. A large
discrepancy means a Colab-side bug (dtype, frame decoding, image resize path, transformers version), so
this cell **stops the notebook** rather than let a long sweep run on a broken pipeline. Not bit-exact
is expected (GPU vs CPU kernels).

In [ ]:
# @title 7. Pipeline validation gate
qwen_spec = next((s for s in specs if s.key == "qwen2_5_vl_3b"), None)
GATE_APPLICABLE = (qwen_spec is not None and not USE_MOCK_BACKENDS and FRAMES_PER_SESSION == 6
                   and {"baseline", "oriented"} <= set(PROMPT_VARIANTS_TO_RUN) and MAX_NEW_TOKENS == 24)
store = cs.CheckpointStore(os.path.join(RESULTS_DIR, f"checkpoint_{RUN_LABEL}.json"), meta=ENV_META)
if not GATE_APPLICABLE:
    print("Validation gate not applicable here (needs the real Qwen candidate, FRAMES_PER_SESSION=6, MAX_NEW_TOKENS=24, "
          "baseline+oriented variants, no mock) - skipping.")
else:
    st = cs.run_sweep([qwen_spec], frames, ["baseline", "oriented"], store, tolerance_mm=TOLERANCE_MM,
                      max_new_tokens=MAX_NEW_TOKENS, hf_token=HF_TOKEN)
    if st.get(qwen_spec.key) not in ("complete", "nothing_to_do"):
        raise RuntimeError(f"Qwen did not complete ({st}); errors: {[e['error'][:200] for e in store.errors[-3:]]}")
    report = cs.validate_against_reference(store, qwen_spec, frames, REFERENCE_JSON, max_new_tokens=MAX_NEW_TOKENS)
    cs.print_validation_report(report)
    if not report["passed"] and not FORCE_CONTINUE_IF_VALIDATION_FAILS:
        raise cs.PipelineValidationError(
            "Colab's Qwen output does not match the local reference closely enough. Do NOT trust a long run. "
            "Suspects, in order: (1) dtype - try DTYPE='bf16'; (2) transformers version / processor resize "
            "(the model-input-size check above); (3) frame decoding. Inspect a frame with cs.render_overlay. "
            "Set FORCE_CONTINUE_IF_VALIDATION_FAILS=True only to proceed knowingly.")

## 8. Full sweep (checkpointed)

In [ ]:
# @title 8. Full sweep: all runnable candidates x all prompt variants (safe to interrupt; re-run to resume)
store = cs.CheckpointStore(os.path.join(RESULTS_DIR, f"checkpoint_{RUN_LABEL}.json"), meta=ENV_META)
sweep_specs = [s for s in specs if "RUNNABLE" not in globals() or s.key in RUNNABLE]
print("sweeping:", [s.key for s in sweep_specs])
t0 = time.time()
try:
    sweep_status = cs.run_sweep(sweep_specs, frames, PROMPT_VARIANTS_TO_RUN, store, tolerance_mm=TOLERANCE_MM,
                                max_new_tokens=MAX_NEW_TOKENS, hf_token=HF_TOKEN)
except KeyboardInterrupt:
    print("\nInterrupted. Everything finished so far is on Drive; re-run this cell to resume.")
    raise
print(f"\nsweep finished in {(time.time()-t0)/60:.1f} min:", sweep_status)

## 9. Results

In [ ]:
# @title 9a. Headline table: candidate x variant
table = cs.aggregate(store, specs, frames, PROMPT_VARIANTS_TO_RUN, TOLERANCE_MM, MAX_NEW_TOKENS)
display(table)
print('''
How to read this (do not skip):
 * n = 60 frames, 10 sessions. A hit-rate difference under ~10 points is within noise at this n; per-session spread (9c) matters more than the pooled number.
 * The [ref] rows are no-model yardsticks on the same frames. A candidate is only doing something if it clearly beats them.
 * Native-API rows ('prompt ignored') do not depend on the prompt variant by construction and ran once.
 * 'dtype' and 'status' say what actually ran; anything other than complete means partial/failed - see 9d.
''')

In [ ]:
# @title 9b. Coordinate-space diagnostic (a wrong coordinate convention shows up as another column being far better)
display(cs.aggregate_coord_space(store, specs, frames, PROMPT_VARIANTS_TO_RUN, TOLERANCE_MM, MAX_NEW_TOKENS))
print("Headline numbers use each backend's DECLARED convention (Qwen: resized model-input space; the others: raw frame pixels, "
      "with PaliGemma2/Moondream2 normalized outputs already denormalized). Choosing the best column per model after the fact "
      "would be forking paths - use this only to spot a mismatch, then fix the backend and re-run.")

In [ ]:
# @title 9c. Per-session breakdown (mean error mm + hits/6) per candidate
for s in specs:
    if any(store.has(cs.item_key(s, v, f, MAX_NEW_TOKENS)) for v in cs.spec_variants(s, PROMPT_VARIANTS_TO_RUN) for f in frames):
        print(f"\n### {s.key}")
        display(cs.per_session_table(store, s, frames, PROMPT_VARIANTS_TO_RUN, MAX_NEW_TOKENS, TOLERANCE_MM))

In [ ]:
# @title 9d. Failures, skips and the full error record
display(pd.DataFrame([{"candidate": k, **{kk: vv for kk, vv in v.items() if kk in ("status", "detail", "dtype", "load_time_s")}}
                      for k, v in store.stages.items()]))
for e in store.errors[-10:]:
    print(f"[{e['candidate']}] {e['stage']} {e.get('variant', '')} {e.get('frame', '')}: {e['error_type']}: {e['error'][:400]}")
print("\n(full tracebacks: the 'errors' list in the checkpoint / exported JSON)")

In [ ]:
# @title 9e. Look at individual predictions (green circle = true target, red x = model answer; blue + = as-parsed point when it differs)
CAND, VARIANT = specs[0].key, PROMPT_VARIANTS_TO_RUN[0]      # <- change these
spec = next(s for s in specs if s.key == CAND)
import matplotlib.pyplot as plt
picks = frames[::max(1, len(frames)//6)][:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, fr in zip(axes.ravel(), picks):
    ax.imshow(cs.render_overlay(fr, store.items.get(cs.item_key(spec, VARIANT, fr, MAX_NEW_TOKENS)))); ax.axis("off")
fig.suptitle(f"{CAND} / {VARIANT}"); plt.show()

In [ ]:
# @title 9f. Export: per-candidate JSON in the local scorer's format + one combined file + CSV
combined_path = cs.export_all(store, specs, frames, PROMPT_VARIANTS_TO_RUN, RESULTS_DIR, TOLERANCE_MM, MAX_NEW_TOKENS, run_label=RUN_LABEL)
print("PASTE BACK / COMPARE THIS FILE:", combined_path)
print("also in", RESULTS_DIR, ":", sorted(f for f in os.listdir(RESULTS_DIR) if RUN_LABEL in f))
if IN_COLAB:
    from google.colab import files
    files.download(combined_path)

## Notes for interpreting / extending

* **Baselines matter.** With a 187.5 x 142 mm platform and targets ~30 mm from the centre, a constant "centre" guess already scores ~25 mm mean error / 15 % hits at 20 mm tolerance; uniform-random points score ~67 mm. The `[ref]` rows compute these on the same frames.
* **Per-session spread, not one pooled number.** Report cell 9c alongside 9a; do not rank candidates from a single run at n = 60 (project rule: `model-iteration-constraints`).
* **Prompt variants are not equally meaningful for every model.** PaliGemma2 (`:detect`) and Moondream2 (`:point`) have native APIs that ignore the prompt; only `:prompt` / `:query` respond to prompt wording.
* **Possible follow-up (not built):** a millimetre-output variant (ask the model for platform coordinates instead of pixels), which would shift the pixel-to-mm burden onto the model.
* **Bugs found here belong in the repo**, not in a notebook cell: fix `core/vlm_backends.py` / `deployment/colab_sweep.py`, rebuild the bundle with `stage_arm2_sweep_for_colab.py`, re-upload.